In [2]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import math
import torch
import random
from shapely import wkt
from shapely.geometry import LineString, Point
import ast
# grafik
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# limit artırma
import sys
sys.setrecursionlimit(2000)
# Limiti 2000'e çıkarır
# özyineleme hata ayıklama içi
import pysnooper

In [3]:
data=pd.read_csv('kocaeli_ana_yollar_final_v15.csv')
print(data.head())

   trip_id      trip_name  segment_id         osm_segment_id  length_meters  \
0        1  Izmit - Gebze        1001  5661714310-9449574131      23.951446   
1        1  Izmit - Gebze        1002  9449574131-6520959031      11.933984   
2        1  Izmit - Gebze        1003  6520959031-2363062636      17.363631   
3        1  Izmit - Gebze        1004  2363062636-1295374326      24.627443   
4        1  Izmit - Gebze        1005   1295374326-968511694     177.067344   

      road_type  lanes lanes_kaynagi  tunnel  max_speed   hiz_kaynagi  \
0       primary      2    varsayilan       0         50           osm   
1  primary_link      2           osm       0         50    varsayilan   
2       primary      2           osm       0         75  yumusatilmis   
3       primary      2    varsayilan       0         90    varsayilan   
4       primary      3           osm       0         90    varsayilan   

   has_traffic_light                                             coords  \
0          

In [4]:
def anomaly_definition(anomaly_type,anomaly_time,fnl_tm):
    anomaly_typs=[1,2,3,4,5,6,7,8,9]
    anomaly_stts=[0,1]
    stts_wghs=[92,8]
    typs_wghs=[30,10,25,1,14,4,4,4,8]
    """
    1- hız sınırı aşımı
    2-olması gerekenden fazla bekleme?
    3-yüksek ivme
    4-kaza
    5-yanlış açı
    6-gps(hız)
    7-gps(ivme)
    8-gps(açı)
    9-gps(donma)
    """
    
    if fnl_tm<=0:
        anomaly_type=0
    if anomaly_type==0:
        anomaly_prb=random.choices(anomaly_stts, weights=stts_wghs, k=1)[0]
        anomaly_type=anomaly_prb
        anomaly_time=0
    else:
        return anomaly_type,anomaly_time,fnl_tm
    if anomaly_prb==1:
        anomaly_type=random.choices(anomaly_typs, weights=typs_wghs, k=1)[0]
        match anomaly_type:
            case 1:
                anomaly_time=np.random.lognormal(mean=np.log(10),sigma=.3)
                anomaly_time=np.clip(anomaly_time,1,17)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 2:
                anomaly_time=np.random.lognormal(mean=np.log(150),sigma=.4)
                anomaly_time=np.clip(anomaly_time,60,600)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 3:
                anomaly_time=np.random.normal(loc=2,scale=1)
                anomaly_time=np.clip(anomaly_time,1,5)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 4:
                accdnt_svrty = np.random.choice(['will_cnteu', 'will_rec', 'cant_cnteu'], p=[0.45, 0.50, 0.05])
                if accdnt_svrty=='will_cnteu':
                    anomaly_time=np.random.normal(loc=150,scale=40)
                if accdnt_svrty=='will_rec':
                    anomaly_time=np.random.lognormal(mean=np.log(750),sigma=.2)
                    anomaly_time=np.clip(anomaly_time,550,1450)
                if accdnt_svrty=='cant_cnteu':
                    anomaly_time=np.random.normal(loc=3000,scale=500)
                    anomaly_time = np.clip(anomaly_time, 1500, 5000)
                anomaly_time=int(anomaly_time)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 5 | 6 | 7 | 8: 
                anomaly_time = np.random.randint(1,5)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
            case 9:
                anomaly_time=np.random.randint(5,16)
                fnl_tm=anomaly_time
                return anomaly_type,anomaly_time,fnl_tm
    else:
        return anomaly_type,anomaly_time,fnl_tm


In [10]:
def distance_calc(avrgspd):
    return (avrgspd/3.6) 
def bearing_calc(dist,lng,geometry,bearing,anomaly_type,anomaly_time,fnl_tm):
    geometry.interpolate()
    geometry_shp=LineString(geometry)
    if lng<0:
        return bearing
    if lng<dist:
        dist = lng - 0.001
        rt=dist/lng
    else:
        rt=dist/lng
    if anomaly_type==4 and anomaly_time>1450 and anomaly_time-5>=fnl_tm:
         return bearing
    rt_crd= geometry_shp.interpolate(rt, normalized=True)
    rt_crd_bfr=geometry_shp.interpolate(rt+0.001, normalized=True)
    x_1, y_1=rt_crd.x , rt_crd.y
    x_2, y_2=rt_crd_bfr.x , rt_crd_bfr.y
    d_x =x_2 - x_1
    d_y = y_2 - y_1
    rdn = math.atan2(d_x, d_y)
    degrs= math.degrees(rdn)
    bearing=(degrs + 360) % 360
    d=np.random.lognormal(mean=np.log(1.4),sigma=.35)
    opr=np.random.choice([-1, 1])
    bearing=bearing+d*opr
    if anomaly_type==4 and anomaly_time-3<fnl_tm:
        wrong_brng=np.random.normal(loc=24,scale=8)
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if anomaly_type==4 and anomaly_time-8>=fnl_tm:
        wrong_brng=5
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if anomaly_type==4 and anomaly_time>1450 and anomaly_time-5<fnl_tm:
        bearing=np.random.normal(loc=180,scale=60)
    if anomaly_type==5:
        wrong_brng=np.random.normal(loc=25,scale=5)
        opr=np.random.choice([-1, 1])
        bearing=bearing+wrong_brng*opr
    if bearing<0:
        return bearing_calc(dist,lng,geometry,bearing,anomaly_type,anomaly_time,fnl_tm)
    return bearing 
def sentetic_statue(statue,traffic):
    traffic=traffic*10
    stts=[1,2,3]
    I1=[[95,2,3],
      [25,70,5],
      [20,4,71]]
    I2=[[20,10,70,0],
      [10,40,50,0],
      [5,5,40,50],
        [0,20,0,80]]
    a,b=0,0 
    t=traffic
    lnr=a*t-b
    I3=[[0,0,0],[0,0,0],[0,0,0]]
    for x in range(0,3):
        for y in range(0,3):
            a=(I2[x][y]-I1[x][y])/(70-30)
            b=I1[x][y]-a*30
            I3[x][y]=a*t+b
    if traffic<=30: 
        if statue==1:
            wghs=I1[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I1[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I1[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
    if 30<traffic<70:
        if statue==1:
            wghs=I3[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I3[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I3[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
    if traffic>=70:
        stts=[1,2,3,4]
        if statue==1:
            wghs=I2[0]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==2:
            wghs=I2[1]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==3:
            wghs=I2[2]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
        if statue==4:
            wghs=I2[3]
            statue_new=random.choices(stts, weights=wghs, k=1)[0]
            return statue_new
 
def accel_calc(pr_spd,speed):
    return (speed-pr_spd)/3.6
     
def sentetic_tarffic_ratio(traffic):
    mean=np.random.normal(loc=2.9, scale=.5)
    traffic_new= np.random.lognormal(mean=np.log(mean),sigma=1.1)
    traffic_new= np.clip(traffic_new, 1, 9)
    if -3.5> traffic-traffic_new or traffic-traffic_new >3.5:
        return sentetic_tarffic_ratio(traffic)
    return traffic_new
def sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type):
    if anomaly_type==1 and traffic>=7:
        anomaly_type=0
    avgspeed=(maxspeed-maxspeed*.08)*(1-(traffic/10)**4)
    if anomaly_type==2 and speed>=10:
        statue=3
    if anomaly_type==2 and speed<10:
        statue=4
    if statue==4:
        d=np.random.normal(loc=0, scale=0.33)
        d=np.clip(d,.01,.89)
        speed_new=0+d
        accl=accel_calc(speed,speed_new)
        
        if 1.5<accl or accl<-3:
            statue=3
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
        return speed_new,accl,statue,anomaly_type
    if speed/avgspeed<.6:
            statue=2
    if avgspeed-speed<-25:
            statue=3
    if anomaly_type==1 and speed>(maxspeed*1.1+5):
        statue=3
    if anomaly_type==1 and speed<(maxspeed*1.1):
        statue=2
    if statue==1:
        sigma=.5*speed/100
        d=np.random.normal(loc=sigma/2, scale=sigma)
        speed_new=speed+d
        accl=accel_calc(speed,speed_new)
        if speed_new<0:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
        while speed_new>(maxspeed*1.1):
            print("hız: speed",speed,"yeni hız: ",speed_new,"durum:",statue,"hız sınırı",maxspeed,"anomaly:",anomaly_type)
            if anomaly_type==1 and speed_new<(maxspeed*1.1+8):
                return speed_new,accl,statue,anomaly_type
            statue=3
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
        return speed_new,accl,statue,anomaly_type

    if statue==2:
        if anomaly_type==3:
            d=np.random.normal(loc=7.2,scale=.4)
            speed_new=speed+d
        else: 
            d=np.random.normal(loc=3,scale=.8)
            d=np.clip(d,1.5,5.4)
            speed_new=speed+d
        
        if speed>=speed_new:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
    if statue==3:
        if anomaly_type==3:
            d=np.random.normal(loc=16.2,scale=1.8)
            d=np.clip(d,10.8,21.6)
            speed_new=speed-d
        else:
            d=np.random.normal(loc=5.75,scale=1.5)
            speed_new=speed-d
        if speed<=speed_new:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
    accl=accel_calc(speed,speed_new)
    if anomaly_type==3:
         while 2.5<accl or accl<-6 or -3<accl<1.5:
             
             return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
         return speed_new,accl,statue,anomaly_type
    else: 
        while 1.5<accl or accl<-3:
            return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
    while speed_new>(maxspeed*1.1):
        print("hız: speed",speed,"yeni hız: ",speed_new,"durum:",statue,"hız sınırı",maxspeed,"anomaly:",anomaly_type)
        if anomaly_type==1 and speed_new<(maxspeed*1.1+8):
            return speed_new,accl,statue,anomaly_type
        statue=3
        return sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
        
    return speed_new,accl,statue,anomaly_type


def sentetic_speed_accident(speed,anomaly_time):
    if anomaly_time<=550:
        if speed<=10:
            speed_new=0.1
        else:
            speed_new=np.random.lognormal(mean=np.log(speed-speed*.35),sigma=0.02)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl
        
    if 550<anomaly_time<=1450:
        if speed<=10:
            speed_new=0.1
        else:
            speed_new=np.random.lognormal(mean=np.log(speed-speed*.55),sigma=0.02)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl
        
    if 1450<anomaly_time:
        if speed<10:
            speed_new=0.1
        speed_new=np.random.lognormal(mean=np.log(speed-speed*.9),sigma=0.02)
        speed_new=np.clip(speed_new,.1,150)
        accl=accel_calc(speed,speed_new)
        return speed_new,accl

def percent_calc(lng,fnl_lng):
    fnl_lng_p=fnl_lng    
    if fnl_lng<0:
        fnl_lng_p=1
    way_rat=100-fnl_lng_p/lng*100
    return way_rat

In [ ]:
def sentetic_data_main(data):
    anomaly_type,anomaly_time,fnl_tm,fnl_lng,time,cntr,accel,pr_mxspd,maxspeed,anomaly_type_t=0,0,0,0,0,0,0,0,0,0
    finalData=[]         
    i=0
    traffic,statue=1,1
    speed=.2
    for trip in data['trip_id']:
        j,bearing,anml_cntr,c=0,0,0,0
        segment=data['segment_id'][i]
        pr_mxspd=maxspeed
        maxspeed=data['max_speed'][i]
        source=data["lanes_kaynagi"][i]
        """if i<(len(data)-1):
            nxt_mxspd=data['max_speed'][i+1]
            nxt_src=data["lanes_kaynagi"][i+1]"""
        for nx in range(3):
            if i<(len(data)-1):
                nxt_mxspd=data['max_speed'][i+1]
                nxt_src=data["lanes_kaynagi"][i+1]
            if (i+nx)<(len(data)-1) and data['length_meters'][i+nx]<150 and (nxt_mxspd-maxspeed)<10:
                nxt_mxspd=data['max_speed'][i+nx]
                nxt_src=data["lanes_kaynagi"][i+nx]
                
            
        geometry=data['coords'][i]
        geometry= wkt.loads(geometry)
        geometry= list(geometry.coords)
        tunnel= data["tunnel"][i]
        j+=1
        if i>=1:
            if data['trip_id'][i]!=data['trip_id'][i-1]:
                anomaly_type,anomaly_time,fnl_tm,fnl_lng,time,cntr,accel,anomaly_type_t,bearing,speed,dwel_duration,dwel_flag=0,0,0,0,0,0,0,0,0,0,0,0
                if kt<1:
                    kt=0
        print("trip: ",trip,"segment",segment)
        lng=data['length_meters'][i]+fnl_lng
        i+=1
        if lng<=0:
            fnl_lng=lng
            continue
        fnl_lng=lng
        
        kt=1
        while kt==1:
            
            if tunnel:
                anomaly_type=9
                anomaly_time=999
                fnl_tm=999
            else:
                anomaly_type=0
                anomaly_time=0
                fnl_tm=0
            
            time+=1
            #print("fark",pr_mxspd-maxspeed,"uzunluk",lng-fnl_lng,"lng",lng,"fnl_lng",fnl_lng)
            #and pr_mxspd-maxspeed>10
            
            anomaly_type,anomaly_time,fnl_tm=anomaly_definition(anomaly_type,anomaly_time,fnl_tm)
            anomaly_type_t=anomaly_type
            if time%30==0:
                traffic=sentetic_tarffic_ratio(traffic)
            statue=sentetic_statue(statue,traffic)
            pr_spd=speed
            if source!='osmn' and (pr_mxspd-maxspeed)>10:
                maxspeed=int((pr_mxspd+maxspeed)/2)+10
                
                    
            if fnl_lng<300 and (nxt_mxspd*1.1)<pr_spd:
                print("hız düşürme ----","trip",trip,"segment_is: ", segment,"mesafe:",fnl_lng,"speed",speed,"next maxspeed:",nxt_mxspd)
                statue=3
                
            
            if anomaly_type==9 and anomaly_time==fnl_tm:
                c=-1
                crnt_spd=finalData[c]['speed']
                crnt_accl=finalData[c]['accel']
                crnt_brng=finalData[c]['bearing']
                crnt_prcnt=finalData[c]['percent']
            if anomaly_type!=4:
                if anomaly_type==1:
                    print("------------------------------------------------------------------------------------>",anomaly_type)
                speed,accel,statue,anomaly_type =sentetic_speed(maxspeed,speed,statue,traffic,anomaly_type)
            else:
                speed,accel=sentetic_speed_accident(speed,anomaly_time)
                fnl_tm-=1
                if fnl_tm==0 and 1450<anomaly_time:
                    break  
                elif fnl_tm==0:
                    anomaly_type=0
                    
            anomaly_type_t=anomaly_type
            if anomaly_type==1:
                if speed>(maxspeed*1.1):
                    fnl_tm-=1
                if speed<=(maxspeed*1.1):
                    anomaly_type_t=0
            if  anomaly_type==2:
                if speed<1.5:
                    anml_cntr+=1
                    if anml_cntr>30:
                        anomaly_type_t=2
                        fnl_tm-=1
                    else:
                        anomaly_type_t=0
                else:
                    anomaly_type_t=0
            if anomaly_type==3:
                if -3>accel>-6 or 2.5<accel<1.5:
                    fnl_tm-=1
                else:
                    anomaly_type_t=0
                    
                    
            dist=distance_calc(speed)
            fnl_lng=lng-dist
            geometry_df=pd.DataFrame(geometry, columns=['x', 'y'])
            bearing=bearing_calc(dist,lng,geometry_df,bearing,anomaly_type,anomaly_time,fnl_tm)
            if anomaly_type==5:
                fnl_tm-=1
            if anomaly_type==6:
                speed=np.random.normal(loc=270.1,scale=90)
                while abs(speed-pr_spd)<60:
                    speed=np.random.normal(loc=270.1,scale=90)
                fnl_tm-=1
            if anomaly_type==7:
                accel=np.random.normal(loc=10,scale=2)
                x=np.random.choice([-1, 1])
                accel=accel*x
                fnl_tm-=1
            if  anomaly_type==8:
                bearing_new=np.random.normal(loc=180,scale=60)
                while abs(bearing-bearing_new)<30:
                    bearing_new=np.random.normal(loc=180,scale=60)
                bearing=bearing_new
                fnl_tm-=1
            if speed<1.5:
                dwel_duration+=1
            else:
                dwel_duration=0
            if dwel_duration>=30:
                dwel_flag=1
            else:
                dwel_flag=0
                
                
            way_rat=percent_calc(lng,fnl_lng)
            if anomaly_type==9:
                fnl_tm-=1
                finalData.append({
                'trip_id': trip,
                'segment_id': segment,
                'time': time,
                'speed': crnt_spd,
                'accel': crnt_accl,
                'bearing': crnt_brng,
                'percent':crnt_prcnt,
                'traffic':int(traffic),
                'dwel_duration':dwel_duration,
                'dwel_flag':dwel_flag,
                'anomaly':anomaly_type_t,
                'maxspeed': maxspeed,
                'tunnel': tunnel
                })

            else:
                finalData.append({
                'trip_id': trip,
                'segment_id': segment,
                'time': time,
                'speed': speed,
                'accel': accel,
                'bearing': bearing,
                'percent':way_rat,
                'traffic':int(traffic),
                'dwel_duration':dwel_duration,
                'dwel_flag':dwel_flag,
                'anomaly':anomaly_type_t,
                'maxspeed': maxspeed,
                'tunnel': tunnel
                })
                
            if anomaly_type==6:
                speed=pr_spd
           
            if int(fnl_lng)<=0:
                kt=0
            else:
                lng=fnl_lng
    

    df_sent = pd.DataFrame(finalData)
    print(df_sent.head(60))
    return df_sent  
final_data=sentetic_data_main(data)
final_data.to_csv("sentetic_data.csv",index=True)

trip:  1 segment 1001
trip:  1 segment 1002
------------------------------------------------------------------------------------> 1
trip:  1 segment 1003
trip:  1 segment 1004
trip:  1 segment 1005
------------------------------------------------------------------------------------> 1
trip:  1 segment 1006
trip:  1 segment 1007
trip:  1 segment 1008
trip:  1 segment 1009
trip:  1 segment 1010
trip:  1 segment 1011
------------------------------------------------------------------------------------> 1
------------------------------------------------------------------------------------> 1
------------------------------------------------------------------------------------> 1
trip:  1 segment 1012
trip:  1 segment 1013
------------------------------------------------------------------------------------> 1
------------------------------------------------------------------------------------> 1
trip:  1 segment 1014
trip:  1 segment 1015
hız düşürme ---- trip 1 segment_is:  1015 mesafe: 23.6

In [8]:
%debug

> c:\users\asus\anaconda3\lib\site-packages\ipykernel\iostream.py(649)write()
    647             raise ValueError(msg)
    648 
--> 649         is_child = not self._is_master_process()
    650         # only touch the buffer in the IO thread to avoid races
    651         with self._buffer_lock:



ipdb>  p trip_id


*** NameError: name 'trip_id' is not defined


ipdb>  segment


*** NameError: name 'segment' is not defined


ipdb>  p segment


*** NameError: name 'segment' is not defined


ipdb>  in


*** SyntaxError: invalid syntax


ipdb>  q
